# Step 2 — tag outcome leakage

Assumes the outcome is depression from PHQ-9 (DPQ). Run after `step1_nonfeature.ipynb`; updates `feature_catalog.csv` in place and writes `step2_counts.txt`.

Categories added (`is_feature` = False):

| category | meaning |
|---|---|
| outcome | the PHQ-9 items themselves (DPQ010–DPQ090) and DPQ100, which is asked only when any item is endorsed |
| outcome_proxy | concurrent self-reported mental-health status or mental-health care, which measures the outcome rather than predicting it |

Kept as features but given a `leak_note` (handled when the column is built):

- **PFQ063A–E:** condition codes causing limitation; code 14 = depression/anxiety/emotional problem (CDC codebook PFQ_D and PFQ_J; 13 = cancer)
- **RXQ_RX:** drug-level file; antidepressants must be excluded when aggregating
- **mixed items:** limitation questions that pool physical and mental causes

In [1]:
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "nhanes" / "paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("run this notebook from inside the repository")
sys.path.insert(0, str(ROOT))
from nhanes.paths import WORK  # noqa: E402

In [2]:
OUTCOME = {f"DPQ0{i}0" for i in range(1, 10)} | {"DPQ100"}
OUTCOME_PROXY = {
    "HSQ480": "days mental health not good (past 30 d)",
    "HSQ496": "days felt anxious (past 30 d)",
    "PFD069D": "days limited by depression/anxiety/emotional problem",
    "HUQ090": "seen mental health professional (past yr)",
    "RHQ551B": "used hormones for emotional symptoms",
}
LEAK_NOTE = {
    **{f"PFQ063{x}": "code 14 = depression/anxiety/emotional problem; drop that code"
       for x in "ABCDE"},
    "RXDDRGID": "exclude antidepressants when aggregating",
    "HSQ490": "pools physical and mental causes",
    "PFQ059": "pools physical, mental and emotional limitations",
    "PFQ010": "pools physical, mental and emotional limitations",
}

In [3]:
path = WORK / "feature_catalog.csv"
cat = pd.read_csv(path)
cat["leak_note"] = cat.variable.map(LEAK_NOTE).fillna("")
feat = cat.is_feature
out = feat & cat.variable.isin(OUTCOME)
proxy = feat & cat.variable.isin(OUTCOME_PROXY)
cat.loc[out, ["category", "reason"]] = ["outcome", "PHQ-9 outcome item"]
cat.loc[proxy, "category"] = "outcome_proxy"
cat.loc[proxy, "reason"] = cat.loc[proxy, "variable"].map(OUTCOME_PROXY)
cat["is_feature"] = cat.category == "feature"
cat["long_format"] = cat.long_format & cat.is_feature
cat.to_csv(path, index=False)

lab = pd.read_csv(WORK / "variable_labels.csv").merge(
    cat[["module", "variable", "is_feature"]], on=["module", "variable"])
rows = [{"cycle": c, "features": d[d.is_feature].variable.nunique()}
        for c, d in lab.groupby("cycle", sort=False)]
lines = ["(module, variable) pairs by category:", cat.category.value_counts().to_string(), "",
         "unique feature variables per cycle after step 2:",
         pd.DataFrame(rows).to_string(index=False), "",
         "kept with leak_note:",
         cat[cat.leak_note != ""][["module", "variable", "leak_note"]].to_string(index=False)]
(WORK / "step2_counts.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

(module, variable) pairs by category:
category
feature           1481
process             74
comment_code        54
id                  49
unit_duplicate      32
weight              22
outcome             10
outcome_proxy        5
design               4

unique feature variables per cycle after step 2:
   cycle  features
 2005-06      1003
 2007-08       969
 2009-10       918
 2011-12       954
 2013-14       998
 2015-16       992
 2017-18       881
2017-20P       678

kept with leak_note:
module variable                                                      leak_note
   HSQ   HSQ490                               pools physical and mental causes
   PFQ   PFQ010               pools physical, mental and emotional limitations
   PFQ   PFQ059               pools physical, mental and emotional limitations
   PFQ  PFQ063A code 14 = depression/anxiety/emotional problem; drop that code
   PFQ  PFQ063B code 14 = depression/anxiety/emotional problem; drop that code
   PFQ  PFQ063C code 14 = dep